# Sesión 02 — Modulación Digital y Análisis de BER
### Laboratorio Python · Sistemas de Comunicaciones Inalámbricas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/02-digital-modulation/lab.ipynb)

## Objetivos del Laboratorio

1. Implementar la cadena TX-RX completa: generación de bits → modulación → canal AWGN → detección → conteo de errores
2. Verificar experimentalmente que la BER simulada coincide con la fórmula teórica $Q(\sqrt{2\gamma_b})$ para BPSK
3. Visualizar cómo el ruido dispersa los puntos de la constelación y cómo aumenta el solapamiento al subir M
4. Cuantificar la ganancia del Gray coding sobre el etiquetado binario natural en 16-QAM
5. Implementar un selector de MCS basado en SNR y calcular el caudal resultante

In [ ]:
%pip install numpy scipy matplotlib --quiet

import numpy as np
from scipy.special import erfc
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

rng = np.random.default_rng(seed=42)
print('Setup completado.')

## Repaso Teórico

### Cadena TX-RX digital

La cadena básica de transmisión digital tiene tres pasos:
1. **Mapeado de bits a símbolos**: $k$ bits → 1 símbolo complejo $s \in \mathcal{C}$
2. **Canal AWGN**: $y = s + n$, donde $n \sim \mathcal{CN}(0, N_0)$ (gaussiano complejo)
3. **Detección ML** (mínima distancia): $\hat{s} = \arg\min_{s_m \in \mathcal{C}} |y - s_m|^2$

Para comparar con la teoría, la relación entre $E_b/N_0$ y la varianza de ruido es:
$$N_0 = \frac{E_s}{k \cdot \gamma_b} = \frac{E_s}{k \cdot 10^{E_b/N_0[\text{dB}]/10}}$$
donde $E_s = \mathbb{E}[|s|^2]$ es la energía media por símbolo y $k = \log_2 M$.

### Fórmulas de BER teórica

- **BPSK/QPSK**: $P_b = Q(\sqrt{2\gamma_b})$
- **M-QAM** (Gray coding): $P_b \approx \frac{4}{\log_2 M}\left(1 - \frac{1}{\sqrt{M}}\right)Q\!\left(\sqrt{\frac{3\log_2 M}{M-1}\gamma_b}\right)$

donde $Q(x) = \frac{1}{2}\,\mathrm{erfc}(x/\sqrt{2})$.

### Implementación de la constelación QAM

Para una constelación cuadrada $\sqrt{M}\times\sqrt{M}$:
- Coordenadas I y Q en $\{-(\sqrt{M}-1), -(\sqrt{M}-3), \ldots, +(\sqrt{M}-1)\}$
- Normalizar para que $E_s = 1$: dividir por $\sqrt{2(M-1)/3}$
- El etiquetado Gray se construye con el patrón reflect-and-prefix en cada eje

---
## Ejercicio 1 — Cadena TX-RX y BER de BPSK
### ⏱ Tiempo estimado: 15 min

Implementa la cadena TX-RX completa para BPSK y verifica la BER teórica $Q(\sqrt{2\gamma_b})$.

**Esquema BPSK**: los bits `0` y `1` se mapean a los símbolos $s \in \{-1, +1\}$ (energía $E_s = 1$).
El receptor decide $\hat{b} = 0$ si $\mathrm{Re}(y) > 0$, y $\hat{b} = 1$ si $\mathrm{Re}(y) \leq 0$.

In [ ]:
def Q_func(x):
    """Función Q: cola derecha de la gaussiana estándar."""
    return 0.5 * erfc(x / np.sqrt(2))

def simulate_bpsk_ber(EbN0_dB_range, N_bits=200_000):
    """
    Simula la BER de BPSK sobre canal AWGN.

    Parámetros
    ----------
    EbN0_dB_range : array de valores Eb/N0 en dB
    N_bits        : número de bits por punto de SNR

    Retorna
    -------
    ber_sim  : BER simulada
    ber_theo : BER teórica Q(sqrt(2*gamma_b))
    """
    ber_sim  = np.zeros(len(EbN0_dB_range))
    ber_theo = np.zeros(len(EbN0_dB_range))

    for i, EbN0_dB in enumerate(EbN0_dB_range):
        gamma_b = 10 ** (EbN0_dB / 10)   # Eb/N0 lineal

        # --- TX ---
        bits = rng.integers(0, 2, N_bits)       # bits aleatorios: 0 o 1
        symbols = 2 * bits - 1                   # mapeado BPSK: 0->-1, 1->+1

        # --- Canal AWGN ---
        # Con Es = 1 y k = 1: N0 = Es / (k * gamma_b) = 1 / gamma_b
        # La varianza del ruido real (parte I) es N0/2 = 1/(2*gamma_b)
        sigma2 = 1 / (2 * gamma_b)
        noise = rng.normal(0, np.sqrt(sigma2), N_bits)
        received = symbols + noise

        # --- Detector BPSK: umbral en 0 ---
        bits_hat = (received <= 0).astype(int)   # 1 si negativo, 0 si positivo

        # --- Conteo de errores ---
        ber_sim[i]  = np.mean(bits != bits_hat)
        ber_theo[i] = Q_func(np.sqrt(2 * gamma_b))

    return ber_sim, ber_theo


EbN0_dB = np.arange(0, 12.5, 0.5)
ber_sim, ber_theo = simulate_bpsk_ber(EbN0_dB)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(EbN0_dB, ber_theo, 'b-',  lw=2,   label='BER teórica BPSK')
ax.semilogy(EbN0_dB, ber_sim,  'ro--', lw=1.5, label='BER simulada BPSK', markersize=5)
ax.axhline(1e-3, color='gray', ls=':', label='BER = $10^{-3}$')
ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('Ejercicio 1 — BER de BPSK: simulada vs teórica')
ax.legend()
ax.set_ylim(1e-6, 1)
ax.set_xlim(0, 12)
plt.tight_layout()
plt.savefig('figures/ber-bpsk-simulation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Preguntas de reflexión — Ejercicio 1

1. ¿En qué rango de SNR divergen más la BER simulada y teórica? ¿Por qué?
   > *Pista: a BER muy baja, el número de errores observados es pequeño y la estimación estadística es ruidosa.*

2. ¿Qué $E_b/N_0$ (dB) da la BER teórica de $10^{-3}$? Localízalo en la curva y compara con el Ejercicio 1(b) de los apuntes.

3. Si duplicas `N_bits` a 400,000, ¿esperas que la curva simulada sea más o menos suave a BER baja? ¿Por qué?

---
## Ejercicio 2 — Constelaciones en el Plano I-Q
### ⏱ Tiempo estimado: 15 min

Visualiza cómo el ruido AWGN dispersa los puntos de la constelación para QPSK, 16-QAM y 64-QAM.
Compara a $E_b/N_0 = 15$ dB y a $E_b/N_0 = 25$ dB.

In [ ]:
def generate_qam_constellation(M):
    """
    Genera la constelación M-QAM con etiquetado Gray y energía media normalizada a 1.

    Retorna
    -------
    symbols : array complejo (M,) — puntos de la constelación
    labels  : array int (M,) — etiqueta de bits (Gray) de cada punto
    """
    sqM = int(np.sqrt(M))
    # Coordenadas PAM: -(sqM-1), -(sqM-3), ..., +(sqM-1)
    pam = np.arange(-(sqM - 1), sqM, 2, dtype=float)   # longitud sqM

    # Código Gray para los índices 0, 1, ..., sqM-1
    gray = np.array([i ^ (i >> 1) for i in range(sqM)])

    symbols = np.zeros(M, dtype=complex)
    labels  = np.zeros(M, dtype=int)

    k = int(np.log2(M))
    half_k = k // 2

    idx = 0
    for qi, q in enumerate(pam):         # eje Q (fila)
        for ii, i_val in enumerate(pam): # eje I (columna)
            symbols[idx] = i_val + 1j * q
            # Etiqueta: bits de Q en los más significativos, bits de I en los menos
            labels[idx] = (gray[qi] << half_k) | gray[ii]
            idx += 1

    # Normalizar energía media a 1
    E_s = np.mean(np.abs(symbols) ** 2)
    symbols /= np.sqrt(E_s)
    return symbols, labels


def transmit_qam(M, EbN0_dB, N_symbols=3000):
    """Genera N_symbols aleatorios de M-QAM y los pasa por AWGN."""
    const, _ = generate_qam_constellation(M)
    k = int(np.log2(M))

    # Índices aleatorios
    idx_tx = rng.integers(0, M, N_symbols)
    s_tx   = const[idx_tx]                        # símbolos transmitidos (Es=1)

    # Varianza de ruido: con Es=1 y k bits/símbolo, N0 = 1/(k*gamma_b)
    gamma_b = 10 ** (EbN0_dB / 10)
    N0 = 1 / (k * gamma_b)
    noise = (rng.normal(0, np.sqrt(N0 / 2), N_symbols) +
             1j * rng.normal(0, np.sqrt(N0 / 2), N_symbols))
    y = s_tx + noise
    return const, s_tx, y


fig, axes = plt.subplots(2, 3, figsize=(14, 9))
configs = [(4, 15), (16, 15), (64, 15), (4, 25), (16, 25), (64, 25)]
titles  = ['QPSK', '16-QAM', '64-QAM']

for ax, (M, snr) in zip(axes.flat, configs):
    const, s_tx, y = transmit_qam(M, snr, N_symbols=2000)
    ax.scatter(y.real, y.imag, s=2, alpha=0.4, color='steelblue', label='Recibidos')
    ax.scatter(const.real, const.imag, s=60, marker='x', color='red',
               linewidths=1.5, label='Constelación')
    ax.set_title(f'{titles[(configs.index((M,snr))) % 3]}, $E_b/N_0={snr}$ dB')
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    lim = 1.8
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')

axes[0, 0].legend(loc='upper right', fontsize=8, markerscale=3)
fig.suptitle('Constelaciones recibidas bajo AWGN (fila superior: 15 dB, inferior: 25 dB)', y=1.01)
plt.tight_layout()
plt.savefig('figures/qam-received-constellations.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Preguntas de reflexión — Ejercicio 2

1. En la fila de 15 dB, ¿en qué modulación es más visible que los puntos invaden las regiones de decisión de los vecinos? ¿Qué significa eso para la BER?

2. En 64-QAM a 25 dB, los puntos de la esquina y del borde tienen menos vecinos que los interiores. ¿Esperarías que la tasa de error sea mayor o menor para los puntos de esquina? ¿Por qué?

3. ¿Qué le ocurre a la dispersión de las nubes al pasar de 15 dB a 25 dB? Comprueba que la reducción visual es consistente con la reducción de $\sigma_n \propto 1/\sqrt{\gamma_b}$.

---
## Ejercicio 3 — Curvas de BER Simuladas vs Teóricas
### ⏱ Tiempo estimado: 25 min

Implementa el simulador Monte Carlo de BER para BPSK, QPSK, 16-QAM y 64-QAM.
Compara las curvas simuladas con las fórmulas teóricas.

**Detector de mínima distancia**: para M-QAM, el símbolo detectado es el más cercano en distancia euclídea al punto recibido.

In [ ]:
def ber_theoretical(M, EbN0_dB_range):
    """BER teórica de M-QAM con Gray coding."""
    gamma_b = 10 ** (np.array(EbN0_dB_range) / 10)
    if M == 2:  # BPSK
        return Q_func(np.sqrt(2 * gamma_b))
    k = np.log2(M)
    arg = np.sqrt(3 * k / (M - 1) * gamma_b)
    prefactor = (4 / k) * (1 - 1 / np.sqrt(M))
    return prefactor * Q_func(arg)


def simulate_qam_ber(M, EbN0_dB_range, N_symbols=100_000):
    """BER simulada de M-QAM con detección de mínima distancia."""
    const, labels = generate_qam_constellation(M)
    k = int(np.log2(M))
    ber = np.zeros(len(EbN0_dB_range))

    for i, EbN0_dB in enumerate(EbN0_dB_range):
        gamma_b = 10 ** (EbN0_dB / 10)
        N0 = 1 / (k * gamma_b)

        # TX: símbolos aleatorios
        idx_tx = rng.integers(0, M, N_symbols)
        bits_tx = np.unpackbits(
            labels[idx_tx].astype(np.uint8)[:, None],
            axis=1, count=k, bitorder='big'
        )  # (N_symbols, k)

        # Canal AWGN
        noise = (rng.normal(0, np.sqrt(N0 / 2), N_symbols) +
                 1j * rng.normal(0, np.sqrt(N0 / 2), N_symbols))
        y = const[idx_tx] + noise

        # Detección ML: mínima distancia euclídea
        # Distancia de cada símbolo recibido a cada punto de la constelación
        dist = np.abs(y[:, None] - const[None, :]) ** 2  # (N_symbols, M)
        idx_hat = np.argmin(dist, axis=1)                 # índice del más cercano

        bits_hat = np.unpackbits(
            labels[idx_hat].astype(np.uint8)[:, None],
            axis=1, count=k, bitorder='big'
        )

        ber[i] = np.mean(bits_tx != bits_hat)

    return ber


EbN0_dB = np.arange(0, 22, 1)
modulations = [('BPSK', 2, 'b'), ('QPSK', 4, 'g'),
               ('16-QAM', 16, 'r'), ('64-QAM', 64, 'm')]

fig, ax = plt.subplots(figsize=(10, 6))

for name, M, color in modulations:
    ber_th  = ber_theoretical(M, EbN0_dB)
    ber_sim = simulate_qam_ber(M, EbN0_dB)
    ax.semilogy(EbN0_dB, ber_th,  color=color, lw=2,   ls='-',  label=f'{name} (teórica)')
    ax.semilogy(EbN0_dB, ber_sim, color=color, lw=1.5, ls='--',
                marker='o', markersize=4, label=f'{name} (simulada)')

ax.axhline(1e-3, color='gray', ls=':', lw=1.5, label='BER = $10^{-3}$')
ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('Ejercicio 3 — BER simulada vs teórica: BPSK, QPSK, 16-QAM, 64-QAM')
ax.legend(ncol=2, fontsize=9)
ax.set_ylim(1e-5, 1)
ax.set_xlim(0, 21)
plt.tight_layout()
plt.savefig('figures/ber-vs-snr-qam.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Preguntas de reflexión — Ejercicio 3

1. Lee la penalización de SNR entre BPSK y 16-QAM en la curva a BER = $10^{-3}$. ¿Coincide con los ~3.7 dB calculados en el Ejercicio 3 de los apuntes?

2. ¿Las curvas simuladas y teóricas se superponen mejor a BER alta o baja? ¿Por qué?
   > *Pista: número de errores observados vs incertidumbre estadística.*

3. Añade una curva para 256-QAM. ¿Qué penalización adicional observas sobre 64-QAM?

---
## Ejercicio 4 — Gray Coding vs Binario Natural en 16-QAM
### ⏱ Tiempo estimado: 15 min

Compara la BER de 16-QAM con etiquetado Gray (el estándar) frente a etiquetado binario natural (0, 1, 2, ..., 15 asignados en orden de barrido).

In [ ]:
def simulate_ber_with_labels(M, labels_array, EbN0_dB_range, N_symbols=100_000):
    """BER simulada con etiquetado arbitrario."""
    const_gray, _ = generate_qam_constellation(M)  # posiciones normalizadas
    k = int(np.log2(M))
    ber = np.zeros(len(EbN0_dB_range))

    for i, EbN0_dB in enumerate(EbN0_dB_range):
        gamma_b = 10 ** (EbN0_dB / 10)
        N0 = 1 / (k * gamma_b)

        idx_tx = rng.integers(0, M, N_symbols)
        bits_tx = np.unpackbits(
            labels_array[idx_tx].astype(np.uint8)[:, None],
            axis=1, count=k, bitorder='big'
        )

        noise = (rng.normal(0, np.sqrt(N0 / 2), N_symbols) +
                 1j * rng.normal(0, np.sqrt(N0 / 2), N_symbols))
        y = const_gray[idx_tx] + noise

        dist = np.abs(y[:, None] - const_gray[None, :]) ** 2
        idx_hat = np.argmin(dist, axis=1)

        bits_hat = np.unpackbits(
            labels_array[idx_hat].astype(np.uint8)[:, None],
            axis=1, count=k, bitorder='big'
        )

        ber[i] = np.mean(bits_tx != bits_hat)

    return ber


M = 16
_, gray_labels = generate_qam_constellation(M)
natural_labels = np.arange(M, dtype=int)       # 0, 1, 2, ..., 15 en orden de barrido

EbN0_dB_range = np.arange(6, 18, 1)

ber_gray    = simulate_ber_with_labels(M, gray_labels,    EbN0_dB_range)
ber_natural = simulate_ber_with_labels(M, natural_labels, EbN0_dB_range)
ber_th      = ber_theoretical(M, EbN0_dB_range)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(EbN0_dB_range, ber_th,      'b-',  lw=2,   label='16-QAM teórica (Gray)')
ax.semilogy(EbN0_dB_range, ber_gray,    'go--', lw=1.5, markersize=5, label='16-QAM simulada (Gray)')
ax.semilogy(EbN0_dB_range, ber_natural, 'rs--', lw=1.5, markersize=5, label='16-QAM simulada (binario natural)')
ax.axhline(1e-3, color='gray', ls=':', lw=1)
ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('Ejercicio 4 — Gray coding vs binario natural en 16-QAM')
ax.legend()
ax.set_ylim(1e-5, 1)
plt.tight_layout()
plt.savefig('figures/gray-vs-natural-ber.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Cuantificar la diferencia en SNR a BER = 1e-3
print("Diferencia aproximada de SNR a BER = 1e-3:")
idx_ref = np.argmin(np.abs(ber_th - 1e-3))
snr_gray    = EbN0_dB_range[np.argmin(np.abs(ber_gray    - 1e-3))]
snr_natural = EbN0_dB_range[np.argmin(np.abs(ber_natural - 1e-3))]
print(f"  Gray coding:     ~{snr_gray} dB")
print(f"  Binario natural: ~{snr_natural} dB")
print(f"  Penalización por binario natural: ~{snr_natural - snr_gray} dB")

### Preguntas de reflexión — Ejercicio 4

1. ¿Cuántos dB de penalización introduce el binario natural respecto al Gray coding a BER = $10^{-3}$? ¿Coincide con la predicción cualitativa del Ejercicio 4 de los apuntes?

2. A SNR muy baja (BER $\approx 0.5$), ¿esperas diferencia entre Gray y natural? ¿Por qué?
   > *Pista: a SNR nula, los errores son completamente aleatorios e independientes del etiquetado.*

3. Prueba a construir un etiquetado "adversarial" donde los vecinos más cercanos siempre difieran en el máximo número de bits posibles. ¿Qué BER obtendrías?

---
## Ejercicio 5 — Adaptación de Enlace: Selección de MCS
### ⏱ Tiempo estimado: 20 min

Implementa un selector de MCS simple que, dado el SNR recibido, elige la modulación y
tasa de código que maximiza el caudal manteniendo BER pre-FEC $\leq 10^{-1.5} \approx 3\times10^{-2}$.

Este umbral de BER pre-FEC ($10^{-1.5}$) es representativo del punto de operación
de los códigos LDPC de 5G NR: el código convierte esa BER de entrada en $<10^{-5}$ a la salida.

In [ ]:
# Tabla MCS simplificada (inspirada en 3GPP TS 38.214)
# Cada entrada: (nombre, M, tasa_código, eficiencia_espectral bits/s/Hz)
MCS_TABLE = [
    ('QPSK 1/4',     4,   0.25, 0.50),
    ('QPSK 1/2',     4,   0.50, 1.00),
    ('QPSK 3/4',     4,   0.75, 1.50),
    ('16-QAM 1/2',  16,   0.50, 2.00),
    ('16-QAM 2/3',  16,   0.67, 2.67),
    ('16-QAM 3/4',  16,   0.75, 3.00),
    ('64-QAM 2/3',  64,   0.67, 4.00),
    ('64-QAM 3/4',  64,   0.75, 4.50),
    ('256-QAM 3/4', 256,  0.75, 6.00),
    ('256-QAM 5/6', 256,  0.83, 6.67),
]

BER_PRE_FEC_THRESHOLD = 10 ** (-1.5)   # ~3.16e-2


def select_mcs(SNR_dB):
    """
    Dado el SNR en dB, selecciona el MCS de mayor eficiencia espectral
    cuya BER teórica está por debajo de BER_PRE_FEC_THRESHOLD.
    Retorna (nombre_MCS, eficiencia_espectral) o ('QPSK 1/4', 0.5) como fallback.
    """
    gamma_b_from_snr = lambda M, r, snr_db: 10**(snr_db/10) / (np.log2(M) * r)

    best = None
    for name, M, rate, eta in MCS_TABLE:
        # Eb/N0 efectivo: SNR - 10*log10(k*r)
        gamma_b = gamma_b_from_snr(M, rate, SNR_dB)
        ber = float(ber_theoretical(M, [10 * np.log10(gamma_b)]))
        if ber <= BER_PRE_FEC_THRESHOLD:
            best = (name, eta)
    return best if best else ('QPSK 1/4', 0.50)


# Evaluar sobre rango de SNR
snr_range  = np.arange(-2, 32, 1)
selected_mcs = [select_mcs(s) for s in snr_range]
eta_selected = [m[1] for m in selected_mcs]
name_selected = [m[0] for m in selected_mcs]

# Caudal pico en un canal de 20 MHz
B_MHz = 20
throughput_Mbps = [eta * B_MHz for eta in eta_selected]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.step(snr_range, eta_selected, where='post', color='steelblue', lw=2)
ax1.set_ylabel('Eficiencia espectral (bits/s/Hz)')
ax1.set_title('Adaptación de enlace: eficiencia espectral y caudal vs SNR (B = 20 MHz)')
ax1.set_ylim(0, 7.5)
# Anotar MCS seleccionado
prev = ''
for s, name in zip(snr_range, name_selected):
    if name != prev:
        ax1.axvline(s, color='gray', ls=':', lw=0.8)
        ax1.text(s + 0.3, 0.3, name, fontsize=7.5, rotation=90, va='bottom')
        prev = name

ax2.step(snr_range, throughput_Mbps, where='post', color='darkorange', lw=2)
ax2.set_xlabel('SNR recibido (dB)')
ax2.set_ylabel('Caudal pico (Mbit/s)')
ax2.set_ylim(0, 140)

plt.tight_layout()
plt.savefig('figures/link-adaptation-mcs.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\nEjemplo: SNR = 20 dB → MCS seleccionado: {select_mcs(20)[0]}, "
      f"caudal = {select_mcs(20)[1]*B_MHz:.0f} Mbit/s")

### Preguntas de reflexión — Ejercicio 5

1. ¿A qué SNR se produce la primera transición de QPSK a 16-QAM? ¿Coincide con el umbral de ~8 dB de la tabla de la Sección 6 de los apuntes?

2. El caudal crece en escalones discretos (escalera). ¿Por qué no crece de forma continua? ¿Qué tecnología permite una variación más gradual del caudal con el SNR?
   > *Pista: piensa en la granularidad del MCS y en la codificación de canal adaptativa.*

3. El umbral BER pre-FEC elegido fue $10^{-1.5}$. ¿Qué ocurriría con el caudal si lo bajas a $10^{-3}$ (BER sin codificación de canal)? ¿Por qué los sistemas reales usan el umbral pre-FEC más alto?

---
## Figuras para los Apuntes

Las celdas siguientes generan las figuras que aparecen en `index.md`.

In [ ]:
# Figura: geometría de detección BPSK
from scipy.stats import norm

EbN0_dB_plot = 7   # ~BER 10^{-3}
gamma_b = 10 ** (EbN0_dB_plot / 10)
sigma = np.sqrt(1 / (2 * gamma_b))
sqrt_Eb = 1.0

x = np.linspace(-3.5, 3.5, 1000)
pdf1 = norm.pdf(x, loc=+sqrt_Eb, scale=sigma)  # s1 enviado
pdf2 = norm.pdf(x, loc=-sqrt_Eb, scale=sigma)  # s2 enviado

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, pdf1, 'b-', lw=2, label='Distribución si se envió $s_1 = +\\sqrt{E_b}$')
ax.plot(x, pdf2, 'r-', lw=2, label='Distribución si se envió $s_2 = -\\sqrt{E_b}$')

# Área de error (izquierda de 0 para s1)
x_err = x[x <= 0]
ax.fill_between(x_err, norm.pdf(x_err, loc=+sqrt_Eb, scale=sigma),
                alpha=0.3, color='blue', label='$P(\\text{error}) = Q(\\sqrt{2\\gamma_b})$')

ax.axvline(0, color='k', ls='--', lw=1.5, label='Frontera de decisión ($y=0$)')
ax.annotate('$s_1 = +\\sqrt{E_b}$', xy=(sqrt_Eb, norm.pdf(sqrt_Eb, sqrt_Eb, sigma)*0.5),
            xytext=(sqrt_Eb+0.6, norm.pdf(sqrt_Eb, sqrt_Eb, sigma)*0.7),
            arrowprops=dict(arrowstyle='->', color='blue'), color='blue', fontsize=10)
ax.annotate('$s_2 = -\\sqrt{E_b}$', xy=(-sqrt_Eb, norm.pdf(-sqrt_Eb, -sqrt_Eb, sigma)*0.5),
            xytext=(-sqrt_Eb-1.8, norm.pdf(-sqrt_Eb, -sqrt_Eb, sigma)*0.7),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

ax.set_xlabel('Valor de la muestra $y$')
ax.set_ylabel('Densidad de probabilidad')
ax.set_title(f'Geometría de detección BPSK ($E_b/N_0 = {EbN0_dB_plot}$ dB, $\\sigma = {sigma:.3f}$)')
ax.legend(fontsize=9)
ax.set_xlim(-3.5, 3.5)
plt.tight_layout()
plt.savefig('figures/bpsk-geometry.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Figura: constelaciones QPSK, 16-QAM, 64-QAM con etiquetas Gray
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs = [(4, 'QPSK'), (16, '16-QAM'), (64, '64-QAM')]

for ax, (M, title) in zip(axes, configs):
    const, labels = generate_qam_constellation(M)
    k = int(np.log2(M))
    ax.scatter(const.real, const.imag, s=80, color='steelblue', zorder=3)

    for s, lab in zip(const, labels):
        bits_str = format(lab, f'0{k}b')
        offset = 0.08 if M <= 16 else 0.04
        ax.text(s.real + offset, s.imag + offset, bits_str,
                fontsize=7 if M <= 16 else 5, color='gray', ha='left')

    # Grid de fronteras de decisión
    sqM = int(np.sqrt(M))
    pam = np.sort(np.unique(const.real))
    boundaries = (pam[:-1] + pam[1:]) / 2
    lim = 1.5 * np.max(np.abs(const.real))
    for b in boundaries:
        ax.axvline(b, color='lightgray', ls='--', lw=0.8, zorder=1)
        ax.axhline(b, color='lightgray', ls='--', lw=0.8, zorder=1)

    ax.axhline(0, color='black', lw=0.5, alpha=0.3)
    ax.axvline(0, color='black', lw=0.5, alpha=0.3)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.set_title(f'{title} ({M} puntos, Gray coding)')

plt.suptitle('Constelaciones M-QAM con etiquetado Gray y fronteras de decisión', y=1.02)
plt.tight_layout()
plt.savefig('figures/qam-constellations.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

---
## Conclusiones

En este laboratorio has implementado y verificado experimentalmente los fundamentos de la modulación digital:

1. **BER de BPSK**: la simulación Monte Carlo confirma $Q(\sqrt{2\gamma_b})$ con precisión creciente al aumentar el número de bits. La divergencia a BER baja refleja la incertidumbre estadística, no un error del modelo.

2. **Constelaciones bajo ruido**: el efecto visual del ruido AWGN es una nube de dispersión alrededor de cada punto. Cuanto más grande es M, más se solapan las nubes a igual SNR — ésta es la manifestación geométrica de la penalización de SNR.

3. **BER de M-QAM**: la penalización de ~4 dB por duplicación del orden de modulación se confirma experimentalmente. Las curvas simuladas reproducen las teóricas excepto a BER muy baja, donde la varianza estadística del estimador Monte Carlo es alta.

4. **Gray coding**: la diferencia de BER frente al binario natural confirma la importancia del etiquetado. El impacto es mayor en modulaciones de alto orden donde la distancia de Hamming entre vecinos puede ser grande con etiquetado arbitrario.

5. **Adaptación de enlace**: el selector de MCS muestra el comportamiento en escalera del caudal con el SNR. La brecha entre el umbral pre-FEC ($10^{-1.5}$) y el post-FEC ($10^{-5}$) es la ganancia que aporta el código LDPC — tema de la Sesión 04.

### Próximos pasos

- **Sesión 03 — OFDM**: la modulación M-QAM se aplica en paralelo sobre $N$ subportadoras. El canal frequency-selective da un SNR distinto por subportadora → el MCS óptimo varía.
- **Sesión 04 — Codificación de canal**: los códigos LDPC y Polar operan cerca del límite de Shannon, desplazando las curvas de BER ~5–8 dB hacia la izquierda.
- **Sesión 06 — MIMO**: spatial multiplexing transmite flujos independientes M-QAM simultáneamente, multiplicando el caudal por el número de capas.

### Lecturas adicionales

- Proakis & Salehi, *Digital Communications* — Cap. 4–5: derivación completa de BER para todas las modulaciones
- 3GPP TS 38.214 §5.1: tablas MCS de 5G NR (niveles 0–28 con modulación y tasa de código exactas)